In [ ]:
from datasets import load_dataset

wnut = load_dataset("wnut_17")

In [ ]:
print(wnut["train"][0])
print()
print(wnut["train"].features["ner_tags"].feature.names)

## Tokenization

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [ ]:
def tokenize_and_align_labels(examples):
    # convert labels from list of tokens to list of ids
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        # Map tokens to their respective word.
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
            # Only label the first token of a given word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_wnut = wnut.map(tokenize_and_align_labels, batched=True)
tokenized_wnut

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
import evaluate

seqeval = evaluate.load("seqeval")

In [ ]:
example = wnut["train"][0]
label_list = wnut["train"].features["ner_tags"].feature.names
label_list

In [ ]:
import numpy as np


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, lb) in zip(prediction, label) if lb != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[lb] for (_, lb) in zip(prediction, label) if lb != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels,
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## Training

In [ ]:
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

In [ ]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import get_peft_model, LoraConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Set to True for 4-bit quantization
    bnb_4bit_compute_dtype=torch.float16,  # Compute dtype (can use float16 or float32)
    bnb_4bit_use_double_quant=True,  # Enable double quantization for efficiency
    bnb_4bit_quant_type="nf4",  # Normalized float4 (NF4) quantization type
)
model = AutoModelForTokenClassification.from_pretrained(
    "deepseek-ai/deepseek-llm-7b-base",
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
    quantization_config=bnb_config,
)

# Define the LoRA configuration
lora_config = LoraConfig(
    r=8,  # Rank of the low-rank matrices (you can experiment with this)
    lora_alpha=32,  # Scaling factor for LoRA matrices (adjust as needed)
    lora_dropout=0.1,  # Dropout rate for LoRA layers
    bias="none",  # Choose bias configuration (none/learning)
    task_type="SEQ_CLS",  # Set the task type (adjust based on your task)
)

# Attach LoRA to the model
model_with_lora = get_peft_model(model, lora_config)


In [ ]:
training_args = TrainingArguments(
    output_dir="my_awesome_wnut_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
    # label_names=label_list,
)

trainer = Trainer(
    model=model_with_lora,
    args=training_args,
    train_dataset=tokenized_wnut["train"],
    eval_dataset=tokenized_wnut["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()